# DiffusionGemma — Multi-Pair Translation (CSV-driven)
Zero-shot translation using DiffusionGemma W4A16 (GoedelMachines/diffusiongemma-26B-A4B-w4a16), evaluated on the 6 directions found in `diffusiongemma_wmt_results.csv` (4 languages: English, Russian, Chinese, Japanese — EN↔RU, EN↔ZH, EN↔JA, plus RU→EN).

Instead of re-downloading WMT test sets, this version reads `source` / `reference` pairs directly from the uploaded CSV, runs inference for each `source`, and writes a new CSV in the **same schema** (`direction, idx, source, hypothesis, reference`) with freshly generated hypotheses, plus a per-direction SacreBLEU summary.

**Before running:**
- Accelerator: **GPU T4 x2** — the model card lists ~18-20 GB peak VRAM (conflicts with this project's earlier ~11.5 GB estimate), which may not fit a single 16 GB T4. Using both T4s via `device_map="auto"` gives 32 GB headroom.
- Internet: ON
- Secrets: add `HF_TOKEN` (from huggingface.co → Settings → Access Tokens). You must accept the Gemma Terms of Use on **both** `google/diffusiongemma-26B-A4B-it` and `GoedelMachines/diffusiongemma-26B-A4B-w4a16` before this token can download weights.
- **Input CSV**: upload `diffusiongemma_wmt_results__2_.csv` as a Kaggle Dataset (or place it under `/kaggle/working/`) and set `CSV_PATH` in Cell 3 accordingly.
- Known risk: this repo's Triton kernels are tuned for Blackwell GPUs (RTX 5090 / GB10). T4 is Turing (sm_75) and falls back to an untuned kernel config — could be slow or fail outright. Also, custom `trust_remote_code` MoE models don't always support clean multi-GPU `device_map="auto"` dispatch (the original NF4 attempt hit a 'meta tensor' bug from CPU-offload dispatch — see `plans/session_10_diffusiongemma.md`). **Always run the smoke test (Cell 5) before the full run (Cell 6).**
- **Session budget**: Kaggle GPU sessions cap at ~9-12h with ~30 GPU-hrs/week on free tier. Per-line latency for this model is unbenchmarked (diffusion LMs may need multiple denoising passes per line). Cell 5's timed smoke test tells you roughly how long all directions × their line counts would take — check it before committing to Cell 6. Cell 6 writes results incrementally (row-by-row) so partial progress survives a crash/timeout.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers accelerate sacrebleu tqdm

In [ ]:
!pip install -q --upgrade transformers


In [ ]:
# Cell 2 — Auth + GPU check
import os, torch
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — Load source/reference pairs from the CSV (instead of downloading WMT test sets)
import csv
from pathlib import Path
from collections import defaultdict, OrderedDict

# --- Point this at your uploaded CSV ---
# If you attached it as a Kaggle Dataset, the path will look like:
#   /kaggle/input/<dataset-slug>/diffusiongemma_wmt_results__2_.csv
# Update as needed.
CSV_PATH = "/kaggle/input/diffusiongemma-wmt-results/diffusiongemma_wmt_results__2_.csv"

# Language code -> full name, used to build the translation prompt
LANG_NAMES = {"en": "English", "ru": "Russian", "zh": "Chinese", "ja": "Japanese"}

def lang_names_for(direction):
    src_code, tgt_code = direction.split("-")
    return LANG_NAMES[src_code], LANG_NAMES[tgt_code]

# Read the CSV and group rows by direction
by_direction = defaultdict(list)
with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        by_direction[row["direction"]].append(row)

# direction -> (src_lines, ref_lines, src_lang, tgt_lang), sorted by idx so
# hypothesis rows line up with source/reference rows later
lines = OrderedDict()
for direction in sorted(by_direction):
    rows = sorted(by_direction[direction], key=lambda r: int(r["idx"]))
    src_lines = [r["source"] for r in rows]
    ref_lines = [r["reference"] for r in rows]
    src_lang, tgt_lang = lang_names_for(direction)
    lines[direction] = (src_lines, ref_lines, src_lang, tgt_lang)
    print(f"{direction} ({src_lang} \u2192 {tgt_lang}): {len(src_lines)} lines")

total_lines = sum(len(v[0]) for v in lines.values())
print(f"\nTotal lines across {len(lines)} directions: {total_lines}")

In [ ]:
from huggingface_hub import snapshot_download
import urllib.request
from transformers import AutoTokenizer, AutoModelForCausalLM

local_dir = snapshot_download(repo_id="GoedelMachines/diffusiongemma-26B-A4B-w4a16", token=HF_TOKEN)
kernels_dir = f"{local_dir}/kernels"

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/nhuvsbu03/diffusiongemma-26B-A4B-w4a16/main/kernels/load_w4_checkpoint.py",
    f"{kernels_dir}/load_w4_checkpoint.py")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/nhuvsbu03/diffusiongemma-26B-A4B-w4a16/main/kernels/fused_moe_w4_v2.py",
    f"{kernels_dir}/fused_moe_w4_v2.py")

tokenizer = AutoTokenizer.from_pretrained(local_dir, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    local_dir,
    trust_remote_code=True,
    device_map="cuda",
    tier="off",
    local_files_only=True,
)
model.eval()
print(f"GPU 0 memory: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")
print(f"GPU 1 memory: {torch.cuda.memory_allocated(1)/1e9:.1f} GB")


In [ ]:
# Cell 5 — Translation helper + timed smoke test (3 lines per pair)
# Run this BEFORE Cell 6 — confirms the model works AND gives a per-line time estimate
# across all 5 pairs before committing to the full ~15,000-line run.
import time

def translate(text, src_lang="English", tgt_lang="Russian", max_new_tokens=150):
    prompt = (
        f"Translate the following {src_lang} sentence to {tgt_lang}. "
        f"Output only the {tgt_lang} translation, nothing else.\n{text}"
    )
    msgs = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        enc = tokenizer.apply_chat_template(
            msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
        )
        input_ids = enc["input_ids"].to("cuda")
    else:
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=max_new_tokens)
    seqs = out.sequences if hasattr(out, "sequences") else out
    full = tokenizer.decode(seqs[0], skip_special_tokens=True)
    for marker in ("model\nthought\n", "model\n"):
        if marker in full:
            full = full.split(marker, 1)[1]
            break
    return full.strip()


print("=== Timed smoke test (3 lines per pair) ===")
per_line_times = []
for pair, (src_lines, ref_lines, src_lang, tgt_lang) in lines.items():
    print(f"\n--- {pair} ({src_lang} → {tgt_lang}) ---")
    for src in src_lines[:3]:
        t0 = time.time()
        hyp = translate(src, src_lang, tgt_lang)
        dt = time.time() - t0
        per_line_times.append(dt)
        print(f"SRC: {src}")
        print(f"HYP: {hyp}")
        print(f"({dt:.1f}s)\n")

avg = sum(per_line_times) / len(per_line_times)
total_lines = sum(len(v[0]) for v in lines.values())
print(f"Avg {avg:.1f}s/line → est. {avg*total_lines/3600:.1f}h for all {total_lines} lines across 5 pairs")

In [ ]:
# Cell 6 — Full inference over all directions, written incrementally to a CSV
# in the same schema as the input (direction, idx, source, hypothesis, reference),
# plus a SacreBLEU score per direction. Run only after Cell 5 confirms good
# output + an acceptable time estimate.
from tqdm.notebook import tqdm

results_root = Path("/kaggle/working/results")
results_root.mkdir(parents=True, exist_ok=True)
out_csv_path = results_root / "diffusiongemma_wmt_results_inferred.csv"
summary_path = results_root / "bleu_summary.txt"

summary = []

with open(out_csv_path, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.writer(out_f)
    writer.writerow(["direction", "idx", "source", "hypothesis", "reference"])

    for direction, (src_lines, ref_lines, src_lang, tgt_lang) in lines.items():
        print(f"\n=== {direction} ({src_lang} \u2192 {tgt_lang}): {len(src_lines)} lines ===")
        hyps = []
        for idx, src in enumerate(tqdm(src_lines, desc=direction)):
            hyp = translate(src, src_lang, tgt_lang)
            hyps.append(hyp)
            writer.writerow([direction, idx, src, hyp, ref_lines[idx]])
        out_f.flush()  # checkpoint after each direction in case the session dies

        bleu = sacrebleu.corpus_bleu(hyps, [ref_lines], tokenize="13a")
        print(f"{direction}: SacreBLEU (13a) = {bleu.score:.2f}  ({len(hyps)} lines)")
        summary.append((direction, bleu.score, len(hyps)))

print("\n=== Summary ===")
for direction, score, n in summary:
    print(f"{direction}: BLEU {score:.2f} ({n} lines)")

with open(summary_path, "w") as f:
    for direction, score, n in summary:
        f.write(f"{direction}: BLEU {score:.2f} ({n} lines)\n")

print(f"\nPer-line results (same schema as input CSV): {out_csv_path}")
print(f"BLEU summary: {summary_path}")